# ARC-AGI-3用共有コード

---

**1. 注意事項**：

Arc-agi-3の概要は以下のURL参照

https://three.arcprize.org/


---

**2. 実装者**：澤田好秀

**3. 実行環境**：Google Colabratory、T4, L4

**4. 参考記事・スライド類**：N/A

---

# セッティング

## インストール各種

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh


In [ ]:
import os
os.environ['PATH'] = '/root/.local/bin:/root/.cargo/bin:' + os.environ['PATH']


In [ ]:
!uv --version


In [ ]:
!pip install arc-agi pillow==11.3.0

## API_KEY設定

In [ ]:
# Please enter your API_KEY if necessary
os.environ['ARC_API_KEY'] = ""
print(os.getenv("ARC_API_KEY"))

# import各種

In [ ]:
import os
import random
import time
from typing import Any
import numpy as np
import sys
import os
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from collections import deque
import hashlib
import collections
import cv2
from typing import List, Dict, Tuple, Optional
import matplotlib.pyplot as plt
from arcengine import FrameData, FrameDataRaw, GameAction, GameState
import arc_agi


# GPU設定

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__, device)

# 自前モデルで実行

### モデル

In [ ]:
class ARCModel(nn.Module):
    """Simple MLP model for ARC-AGI-3 action + coord prediction"""

    def __init__(self, embedding_dim=16, hidden_dim=256):
        super().__init__()

        self.emb = nn.Conv2d(1, embedding_dim, kernel_size=3, padding=1, bias=False)
        flat_dim = 64 * 64 * embedding_dim

        # Encoding
        self.board_mlp = nn.Sequential(
            nn.Linear(flat_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )


        # for action
        self.action_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 7),
        )

        # for ACTION6
        self.coord_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, frame):
        B = frame.size(0)

        x = self.emb(frame.float())
        x = x.view(B, -1)
        h = self.board_mlp(x)

        action_logits = self.action_head(h)
        coords = self.coord_head(h)
        # scaling
        coords = torch.sigmoid(coords) * 63.0

        return action_logits, coords




class Action:
    def __init__(self, device):

        self.device = device

        # Track previous state/action for experience creation
        self.prev_frame = None
        self.prev_action_idx = None

        # Action mapping
        self.action_list = [GameAction.RESET, GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3,
                           GameAction.ACTION4, GameAction.ACTION5, GameAction.ACTION6]

        # Exploration
        self.epsilon = 0.5

        self.action_model = ARCModel(embedding_dim=16, hidden_dim=256).to(self.device)
        self.optimizer = optim.Adam(self.action_model.parameters(), lr=0.0001)

        # Loss functions
        self.action_criterion = nn.CrossEntropyLoss()
        self.coord_criterion = nn.MSELoss()

    def frame_to_tensor(self, frame_array):
        """Convert frame array to tensor"""
        return torch.from_numpy(frame_array)

    def compute_reward(self, frame_changed):

        # 初期化
        reward = 0.0
        if not frame_changed:
            reward -= 2.0

        return reward

    def train(self, frame_tensor, action_id, reward, coords=None, action_counter=0):

        # forwarding
        action_logits, pred_coords = self.action_model(frame_tensor)

        # Action loss
        action_target = torch.tensor([action_id], dtype=torch.long).to(self.device)
        action_loss = self.action_criterion(action_logits, action_target)

        # Coord loss
        coord_loss = torch.zeros(1, device=self.device)
        if coords is not None:
            coord_target = torch.tensor(
                [[coords[0], coords[1]]], dtype=torch.float32
            ).to(self.device)
            coord_loss = self.coord_criterion(pred_coords, coord_target)

        base_loss = action_loss + coord_loss

        if reward > 0:
            total_loss = base_loss
        else:
            total_loss = abs(reward) * base_loss

        self.optimizer.zero_grad()
        total_loss.backward()
        self.optimizer.step()

        if action_counter % 10 == 0:
            print(
                f"Step {action_counter}: Loss={total_loss.item():.4f} "
                f"(action={action_loss.item():.3f}, coord={coord_loss.item():.3f}), "
                f"Reward={reward:.2f}"
              )


    def choose_action(self, obs, action_counter):
        """Choose action using CNN model and learn from previous step"""

        if obs.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            # Reset previous tracking on game reset
            self.prev_frame = None
            self.prev_action_idx = None
            action = GameAction.RESET
            return action

        # Prepare input
        frame_tensor = self.frame_to_tensor(obs.frame[-1]).unsqueeze(0).unsqueeze(0).to(self.device)  # (1, 64, 64)

        # Predict (without action_id, so next_frame_logits will be None)
        with torch.no_grad():
            action_logits, coords = self.action_model(frame_tensor)
            # Get action probabilities
            action_probs = torch.softmax(action_logits[0], dim=0).cpu().numpy()


        # ε-greedy action selection
        if np.random.random() < self.epsilon:
            # Random action from available actions
            predicted_action_id = np.random.choice(len(self.action_list))
        else:
            # Use model prediction
            predicted_action_id = int(np.argmax(action_probs))

        # Convert to GameAction
        action = self.action_list[predicted_action_id]


        # Set data based on action type
        pred_coords = None
        action_data = {}
        if predicted_action_id == 6:
            # Use predicted coordinates with ε-greedy exploration
            if np.random.random() < self.epsilon:
                # Random exploration
                x = np.random.randint(0, 64)
                y = np.random.randint(0, 64)
            else:
                # Use model prediction
                x = int(coords[0, 0].item())
                y = int(coords[0, 1].item())
                x = np.clip(x, 0, 63)
                y = np.clip(y, 0, 63)

            pred_coords = (x, y)
            action_data = {
                "x": int(x),
                "y": int(y),
            }
            print(f"[ACTION6] clicking ({x}, {y})")


        # Learn from previous step if available
        if self.prev_frame is not None and self.prev_action_id is not None:

            # Check if frame changed
            frame_changed = not torch.equal(self.prev_frame, frame_tensor)

            # Compute reward
            reward = self.compute_reward(frame_changed)

            # Train model periodically
            self.train(self.prev_frame, self.prev_action_id, reward, self.prev_coords, action_counter)


        # Store for next learning step
        self.prev_frame = frame_tensor.detach().clone()
        self.prev_action_id = predicted_action_id
        self.prev_coords = pred_coords

        return action, action_data




## 実行

In [ ]:
# Initialize the ARC-AGI-3 client
arc = arc_agi.Arcade()

# prev_games only
target_games = ["ls20", "ft09", "vc33"]

# Play the game
games = arc.get_environments()
for game in games:

    if not any(target in game.game_id for target in target_games):
        continue

    # Initialization
    action_agent = Action(device)
    env = arc.make(game.game_id)
    env.reset()
    for step in range(100):

        obs = env.observation_space
        action, action_data = action_agent.choose_action(obs, step)
        if step % 1 == 0:
            print(f"Game id {obs.game_id} -- {action}: count {step}, levels {obs.levels_completed}, win levels {obs.win_levels}")

        # Perform the action (rendering happens automatically)
        obs = env.step(action, data=action_data)

        # Check game state
        if obs and obs.state == GameState.WIN:
            print(f"Game won at step {step}!")
            break
        elif obs and obs.state == GameState.GAME_OVER:
            env.reset()

# Get and display scorecard
scorecard = arc.get_scorecard()
if scorecard:
    print("\n")
    print(f"Total Levels Completed: {scorecard.total_levels_completed}")
    print(f"Final Score: {scorecard.score}")




# 以上

In [ ]:
#